# Start with the Databases

In [1]:
# Originally outlined in the "Bridge Queries" Notebook
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

We know the server the TIMs database is on to start with, `doticsqlp31`.

We need to figure out what databases it actually contains besides just TIMs.

In [2]:
connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

engine = create_engine(connection_url)
conn = engine.connect()

Since the connection is successful, we know the server is active and we  
have access. Here's a Gemini query to get what we're after (the list of  
tables and databases in the server).

```
I have the following code,

# Originally outlined in the "Bridge Queries" Notebook
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

How can I see what tables are available in this database, and what other databases are available on the server?
```

and Gemini provides this code,

In [3]:
# Create an inspector object
inspector = inspect(engine)

# Get and print the list of tables
tables_in_tims = inspector.get_table_names()
print("--- Tables available in the 'TIMS' database ---")
for table in tables_in_tims:
    print(table)

--- Tables available in the 'TIMS' database ---
Datasets
Fields
GDB_CONFLICTS
GDB_EDITINGTEMPLATERELATIONSHIPS
GDB_EDITINGTEMPLATES
GDB_ITEMRELATIONSHIPS
GDB_ITEMRELATIONSHIPTYPES
GDB_ITEMS
GDB_ITEMTYPES
GDB_LOCKS
GDB_REPLICALOG
GDB_TABLES_LAST_MODIFIED
i10
i11
i242
i243
i6
i7
i8
i9
rowcount_table
SDE_archives
SDE_branch_tables_modified
SDE_branches
SDE_column_registry
SDE_compress_log
SDE_dbtune
SDE_geometry_columns
SDE_GEOMETRY1
SDE_layer_locks
SDE_layer_stats
SDE_layers
SDE_lineages_modified
SDE_locators
SDE_logfile_pool
SDE_metadata
SDE_multibranch_tables
SDE_mvtables_modified
SDE_object_ids
SDE_object_locks
SDE_process_information
SDE_raster_columns
SDE_server_config
SDE_spatial_references
SDE_state_lineages
SDE_state_locks
SDE_states
SDE_table_locks
SDE_table_registry
SDE_tables_last_edit_time
SDE_tables_modified
SDE_version
SDE_version_history
SDE_versions
SDE_xml_columns
SDE_xml_index_tags
SDE_xml_indexes


Listing all databases on a server.

In [4]:
databases_on_server = inspector.get_schema_names()
print("\n--- Databases available on the server ---")
for db_name in databases_on_server:
    print(db_name)


--- Databases available on the server ---
db_accessadmin
db_backupoperator
db_datareader
db_datawriter
db_ddladmin
db_denydatareader
db_denydatawriter
db_owner
db_securityadmin
dbo
guest
INFORMATION_SCHEMA
sys
TIMS


There are probably more, and these are just the ones I have access to.

Since I don't know what any of these are, I can ask Gemini to summarize  
which of these it recognizes as like system/default tables,

Gemini's Response:
--- 

**Unlikely to Contain Interesting Data (System Defaults)**  

You can safely ignore the following as they are part of the SQL Server  
system for managing permissions and metadata. They do not contain  
application or business data.

- db_accessadmin
- db_backupoperator
- db_datareader
- db_datawriter
- db_ddladmin
- db_denydatareader
- db_denydatawriter
- db_owner
- db_securityadmin

    - Reason: These are all built-in fixed-database roles. They are used to  
grant permissions to users (e.g., a user added to the db_datareader role  
can read data from tables), but they don't hold tables themselves.

- guest

    - Reason: This is a default user account that exists for access by  
users who don't have a specific account in the database.

- INFORMATION_SCHEMA

    - Reason: This is a system schema that contains views with metadata  
about the database—information about the tables, columns, etc., but not the  
data within them.

- sys

    - Reason: This is the primary system catalog for SQL Server. It holds a  
vast amount of internal metadata and configuration information.
 
---

**Very Likely to Contain Interesting Data**

You should focus your exploration on these schemas, as they are the  
intended containers for the application's data.

- TIMS

    - Reason: This is the name of your database. It's the top-level  
      container for all the schemas and tables related to the "TIMS"  
      application. This is your main target.


- dbo (database owner)

    - Reason: This is the default schema in a SQL Server database. If a  
developer creates a table without specifying a different schema, it will be  
placed in dbo by default. It is extremely common for most or all of an  
application's tables to reside here.

**Summary**: For your next steps, you should investigate the tables within  
the TIMS database, and you will almost certainly find them under the dbo  
schema.

All Tables in TIMs

In [5]:
# This is the key step for your request
tables_in_dbo = inspector.get_table_names(schema='dbo')

print("--- Tables in the 'dbo' schema of the 'TIMS' database ---")
if tables_in_dbo:
    for table_name in tables_in_dbo:
        print(table_name)
else:
    print("No tables found in the 'dbo' schema.")

--- Tables in the 'dbo' schema of the 'TIMS' database ---
Datasets
Fields
GDB_CONFLICTS
GDB_EDITINGTEMPLATERELATIONSHIPS
GDB_EDITINGTEMPLATES
GDB_ITEMRELATIONSHIPS
GDB_ITEMRELATIONSHIPTYPES
GDB_ITEMS
GDB_ITEMTYPES
GDB_LOCKS
GDB_REPLICALOG
GDB_TABLES_LAST_MODIFIED
i10
i11
i242
i243
i6
i7
i8
i9
rowcount_table
SDE_archives
SDE_branch_tables_modified
SDE_branches
SDE_column_registry
SDE_compress_log
SDE_dbtune
SDE_geometry_columns
SDE_GEOMETRY1
SDE_layer_locks
SDE_layer_stats
SDE_layers
SDE_lineages_modified
SDE_locators
SDE_logfile_pool
SDE_metadata
SDE_multibranch_tables
SDE_mvtables_modified
SDE_object_ids
SDE_object_locks
SDE_process_information
SDE_raster_columns
SDE_server_config
SDE_spatial_references
SDE_state_lineages
SDE_state_locks
SDE_states
SDE_table_locks
SDE_table_registry
SDE_tables_last_edit_time
SDE_tables_modified
SDE_version
SDE_version_history
SDE_versions
SDE_xml_columns
SDE_xml_index_tags
SDE_xml_indexes
